### 0. Preliminaries
Import necessary packages

In [776]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

### 1. Model Setup
Define constant parameters, user functions, and simulation individuals

In [777]:
# Define constant parameters
county_size = 20 # miles
taxi_speed = 20 # miles per hour
base_fare = 3.00 # pounds
per_mile_fare = 2.00 # pounds per mile
petrol_cost = 0.2 # pounds per mile

In [778]:
def distance(loc1, loc2):
    return np.sqrt((loc1[0] - loc2[0])**2 + (loc1[1] - loc2[1])**2)

def generate_location():
    return (random.uniform(0, county_size), random.uniform(0, county_size))

def travel_time(d_OD):
    mu = d_OD / taxi_speed
    return random.uniform(0.8*mu, 1.2*mu) 

### 2. Simulation

In [779]:
def run_simulation(T = 100):
    ''' Runs the rideshare Simulation with BoxCar's initial assumptions. 
            Inputs:
            - T: Termination Time (in hours)
            Output:
            - results: dictionary of performance metrics indexed by metric type '''
    # Initialise performance statistics
    N_D = 0
    N_R = 0
    T_D = 0
    T_R = 0
    P = 0
    R = 0    # total completed rides
    matched_rides = 0    # started but dropoff is after TerminationTime - do not count driver earnings
    picked_up_rides = 0    # matched and actually gets picked up before TerminationTime
    abandoned_rides = 0    # abandoned due to patience time being reached
    T_P = []
    T_All = []
    T_W = []

    driver_ride_counts = {}
    driver_idle_time = {}
    driver_last_busy_time = {}
    
    # initialize simulation
    TNOW = 0
    TerminationTime = T #hrs
    EventCalendar = []

    def schedule_event(time, event_type, data=None):
        '''Event scheduler that keeps track of times, event type, and (optionally) user data'''
        EventCalendar.append((time,event_type,data))
   
    drivers =[]
    idle_drivers=[]
    riders = []
    waiting_riders=[]

    driver_id_counter = 0
    rider_id_counter = 0

    schedule_event(random.expovariate(3), 1)   # 1 = Driver Arrival
    schedule_event(random.expovariate(30), 2)  # 2 = Rider Arrival
    schedule_event(TerminationTime, 8)             # 8 = Termination

    while TNOW < TerminationTime:
        TNEXT, TypeNext, Data = min(EventCalendar, key=lambda x: x[0])
        EventCalendar.remove((TNEXT, TypeNext, Data))

        TNOW = TNEXT

        # DriverArrival
        if TypeNext == 1:
            driver_id_counter += 1

            # Generate driver location and offline time (initialize other attributes)
            driver = {
                "id": driver_id_counter,
                "location": generate_location(),
                'offline_time': TNOW + random.uniform(5,8),
                'idle': True,
                'active': True,
                'earnings': 0,
                'online_time': TNOW
            }
            drivers.append(driver)
            
            # Schedule offline time
            schedule_event(driver['offline_time'], event_type=7, data = driver)

            # Match driver to waiting rider
            if waiting_riders:
                # match closest waiting rider
                rider = min(waiting_riders, key = lambda r:distance(driver['location'], r['origin']))
                waiting_riders.remove(rider)
                
                # update statuses
                rider['status'] = 'matched'
                driver['idle'] = False
                
                # schedule pickup start
                schedule_event(TNOW, event_type=3, data = (driver, rider)) 

                # update performance metrics
                matched_rides += 1
            else:
                idle_drivers.append(driver)
            
            # Schedule next driver arrival
            schedule_event(TNOW + random.expovariate(3), event_type=1)

            # Update performance metrics
            N_D += 1
            driver['online_time'] = TNOW
            driver_ride_counts[driver['id']] = 0
            driver_idle_time[driver['id']] = 0
            driver_last_busy_time[driver['id']] = TNOW

        # RiderArrival
        elif TypeNext == 2: 
            rider_id_counter += 1

            # Generate rider location and patience time (initialize other attributes)
            rider = {
                "id": rider_id_counter,
                "origin": generate_location(),
                'destination': generate_location(),
                'patience_time': random.expovariate(5),
                'status': 'waiting'
            }
            riders.append(rider)
            
            # Match driver to idle active driver
            idle_active_drivers = [driver for driver in idle_drivers if driver['active']]

            if idle_active_drivers:
                # match to closest driver
                driver = min(idle_active_drivers, key=lambda d: distance(d['location'], rider['origin']))
                idle_drivers.remove(driver)

                # update statuses
                rider['status'] = 'matched'
                driver['idle'] = False

                # schedule PickupStart
                schedule_event(TNOW, event_type=3, data = (driver,rider))

                # update performance metrics
                matched_rides += 1
            else:
                waiting_riders.append(rider)

                #schedule patience deadline for waiting rider
                schedule_event(TNOW+rider['patience_time'], event_type=6, data = rider)
            
            # schedule next rider arrival
            schedule_event(TNOW + random.expovariate(30), event_type=2)

            # Update performance metrics
            N_R += 1
            rider['arrival_time'] = TNOW
            #T_P.append(rider['patience_time'])

        # PickupStart
        elif TypeNext == 3:
            driver,rider = Data

            # in very rare event that driver becomes offline at exact time it gets matched, 
            # add rider back to waiting list
            if not driver['active']:
                rider['status'] = 'waiting'
                waiting_riders.append(rider)
                continue
            
            # Calculate actual travel time 
            d_P = distance(driver['location'],rider['origin'])
            t_P = travel_time(d_P)

            # Schedule Pickup event
            schedule_event(TNOW + t_P, event_type=4, data = (driver,rider,d_P))

            # Update performance metrics
            waiting_time = TNOW - rider['arrival_time']  # time waited between rider arrival and matching
            T_W.append(waiting_time)
            T_All.append(waiting_time)
            T_P.append(rider['patience_time'])
            driver_ride_counts[driver['id']] += 1
            driver_idle_time[driver['id']] += TNOW - driver_last_busy_time[driver['id']]
            driver_last_busy_time[driver['id']] = TNOW
            
        
        # Pickup
        elif TypeNext == 4:
            driver, rider, d_P = Data
            
            # Calculate actual travel time
            d_OD = distance(rider['origin'], rider['destination'])
            t_OD = travel_time(d_OD)

            # Schedule Dropoff
            schedule_event(TNOW + t_OD, event_type=5, data = (driver,rider,d_P,d_OD,t_OD))

            # update performance metrics
            picked_up_rides += 1

        # Dropoff
        elif TypeNext == 5:
            driver, rider, d_P, d_OD,t_OD = Data

            # update rider status
            rider["status"] = "dropped-off"

            # calculate driver earnings
            rider_fare = base_fare + per_mile_fare * d_OD
            driver_cost = petrol_cost*(d_P+d_OD)
            driver_profit = rider_fare - driver_cost
            driver['earnings'] += driver_profit      # changed this to make it cumulative profit
            
            # Update driver information
            driver['location'] = rider['destination']
            driver['idle'] = True

            # Check if driver is meant to be offline now
            if TNOW >= driver['offline_time']:
                driver['active'] = False
            else:
                if waiting_riders:
                    next_rider = min(waiting_riders,
                                    key=lambda r: distance(driver['location'], r['origin']))
                    waiting_riders.remove(next_rider)
                    next_rider['status'] = 'matched'
                    driver['idle'] = False
                    # schedule next pickup if driver gets matched to a new rider
                    schedule_event(TNOW, event_type=3, data = (driver, next_rider))

                    # update performance metrics
                    matched_rides += 1
                else:
                    idle_drivers.append(driver)

            # Update performance metrics
            R += 1
            T_R += t_OD  
            P += driver_profit
            driver_last_busy_time[driver['id']] = TNOW
        
        # RiderAbandonment
        elif TypeNext == 6:
            rider = Data

            # remove abandonded riders and update status
            if rider['status'] == 'waiting':
                waiting_riders.remove(rider)
                rider["status"] = "abandoned"
                abandoned_rides += 1

                # Update performance metrics
                T_All.append(rider['patience_time'])
                T_P.append(rider['patience_time'])
        
        # DriverOffline
        elif TypeNext == 7:
            driver = Data
            
            # remove offline drivers and update status
            if driver["active"] and driver["idle"]:
                driver["active"] = False
                if driver in idle_drivers:
                    idle_drivers.remove(driver)

            # Update performance metrics
            T_D += TNOW - driver['online_time']
            driver_idle_time[driver['id']] += TNOW - driver_last_busy_time[driver['id']]

        # termination
        else:
            # Update performance metrics
            for driver in drivers:
                if driver['active']:
                    T_D += TNOW - driver['online_time']
                    
                    if driver['idle']:
                        driver_idle_time[driver['id']] += TNOW - driver_last_busy_time[driver['id']]

    # combine all metrics to output everything in a single dictionary
    results = {
        "system_metrics": {
            "N_D": N_D,
            "N_R": N_R,
            "R": R,
            "matched_rides": matched_rides,
            "picked_up_rides": picked_up_rides,
            "abandoned_rides": abandoned_rides,
            'TerminationTime': T
        },
    
        "financial_metrics": {
            "P": P
        },

        "time_metrics": {
            "T_D": T_D,
            "T_R": T_R,
            "T_W": T_W,
            "T_P": T_P,
            "T_All": T_All
        },

        "driver_metrics": {
            "driver_ride_counts": driver_ride_counts,
            "driver_idle_time": driver_idle_time,
            "driver_last_busy_time": driver_last_busy_time
        },

        "entities": {
            "drivers": drivers,
            "riders": riders
        }
    }

    return results


In [780]:
def compute_metrics(sim_results):
    '''
    Input: results dictionary returned by run_simulation.
    Outputs the following performance metrics for this instance of the simulation:
    - 'abandonment_rate': 
    - 'avg_wait_matched': 
    - 'avg_wait_all': 
    - 'all_wait_ratios':
    - 'match_wait_ratios':
    - 'avg_PT_wait_all':
    - 'avg_PT_wait_matched':
    - 'avg_earnings_per_hour': 
    - 'avg_driver_profit':
    - 'avg_idle_time_per_ride':
    - 'driver_idle_per_ride':
    - 'idle_proportion':
    '''

    sys = sim_results["system_metrics"]
    time = sim_results["time_metrics"]
    fin = sim_results["financial_metrics"]
    driver_metrics = sim_results["driver_metrics"]
    driver_idle_time = driver_metrics["driver_idle_time"]
    driver_ride_counts = driver_metrics["driver_ride_counts"]

    N_R = sys["N_R"]
    R = sys["R"]
    abandoned = sys["abandoned_rides"]
    T = sys["TerminationTime"]

    T_D = time["T_D"]
    T_R = time["T_R"]
    T_W = time["T_W"]
    T_P = time["T_P"]
    T_All = time["T_All"]

    P = fin["P"]

    drivers = sim_results["entities"]["drivers"]

    # rider abandonments
    abandonment_rate = abandoned / N_R if N_R else 0

    # rider waiting times
    avg_wait_matched = np.mean(T_W) if T_W else 0
    avg_wait_all = np.mean(T_All) if T_All else 0
    # proportions of patience times waited by matched/all riders
    all_wait_ratios = np.array(T_All) / np.array(T_P) if T_P else np.array([])
    match_wait_ratios = all_wait_ratios[all_wait_ratios < 1]
    avg_PT_wait_all = np.mean(all_wait_ratios)
    avg_PT_wait_matched = np.mean(match_wait_ratios)

    # driver earnings
    avg_earnings_per_hour = P / T_D if T_D else 0
    # driver-level profits
    driver_profits = [d["earnings"] for d in drivers]
    avg_profit = np.mean(driver_profits) if driver_profits else 0

    # fairness among drivers
    driver_rides = list(driver_ride_counts.values())

    avg_driver_idle_per_ride = []
    driver_working_ratios = []

    for driver in drivers:
        d_ID = driver["id"]
        rides = driver_ride_counts.get(d_ID, 0)
        idle_time = driver_idle_time.get(d_ID, 0)

        # compute online time
        if driver['active']:
            online_time = T - driver['online_time']
        else:
            online_time = driver['offline_time'] - driver['online_time']

        if rides > 0:
            avg_driver_idle_per_ride.append(idle_time / rides)
        else:
            avg_driver_idle_per_ride.append(0)

        if online_time > 0:
            driver_working_ratios.append((online_time - idle_time) / online_time)
        else:
            driver_working_ratios.append(0)

    avg_rides_per_driver = np.mean(driver_rides)
    avg_working_prop = np.mean(driver_working_ratios)

    # resting times
    avg_idle_time_per_ride = (T_D-T_R) / R if R else 0
    idle_prop = (T_D-T_R) / T_D if T_D else 0

    # return dictionary of computed metrics and arrays for histograms
    return {
        "abandonment_rate": abandonment_rate,
        "avg_wait_matched": avg_wait_matched,
        "avg_wait_all": avg_wait_all,
        'all_wait_ratios': all_wait_ratios,
        'match_wait_ratios': match_wait_ratios,
        'avg_PT_wait_all': avg_PT_wait_all,
        'avg_PT_wait_matched': avg_PT_wait_matched,
        "avg_earnings_per_hour": avg_earnings_per_hour,
        "avg_driver_profit": avg_profit,
        'driver_profits': driver_profits,
        'avg_rides_per_driver': avg_rides_per_driver,
        'avg_working_prop': avg_working_prop,
        'avg_idle_time_per_ride': avg_idle_time_per_ride,
        'driver_idle_per_ride': avg_driver_idle_per_ride,
        "idle_proportion": idle_prop,
    }

In [781]:
import numpy as np
from scipy.stats import t

def compute_CIs(metrics_list, alpha=0.05):
    '''
    Uses replicates of simulation to find confidence intervals for performance metrics 
    returned by 'compute_metrics'.

    Input:
    - 'metrics_list': list of dictionaries, where each element is the output of 
        compute_metrics() from one simulation run.

    'alpha': significance level (default 0.05 to return 95% CIs)

    Output:
    - 'results': the following dictionary:
        metric_name : {
            'mean': sample mean,
            'lower': CI lower bound,
            'upper': CI upper bound,
            'std': sample std deviation
        }
    '''

    results = {}

    keys = metrics_list[0].keys()

    for key in keys:

        # collect values for this metric
        values = [m[key] for m in metrics_list]

        # skip vectors/lists/arrays automatically
        if isinstance(values[0], (list, np.ndarray)):
            continue

        values = np.array(values)

        n = len(values)
        mean = np.mean(values)
        s = np.std(values, ddof=1)

        # estimating standard deviation from simulated data so use student's t critical value
        t_crit = t.ppf(1 - alpha/2, df=n-1)
        pm_part = t_crit * s / np.sqrt(n)

        results[key] = {
        "mean": round(mean, 3),
        "CI_lower": round(mean - pm_part, 3),
        "CI_upper": round(mean + pm_part, 3)
        }

    return results

In [782]:
def run_replications(n_replications=50):
    '''
    Run simulations and output confidence intervals for performance metrics.
    '''
    metrics_list = []

    for _ in range(n_replications):
        sim_results = run_simulation()
        metrics_list.append(compute_metrics(sim_results))

    return compute_CIs(metrics_list)

In [783]:
results = run_replications(100)

In [784]:
results

{'abandonment_rate': {'mean': 0.285, 'CI_lower': 0.279, 'CI_upper': 0.29},
 'avg_wait_matched': {'mean': 0.043, 'CI_lower': 0.042, 'CI_upper': 0.044},
 'avg_wait_all': {'mean': 0.057, 'CI_lower': 0.056, 'CI_upper': 0.058},
 'avg_PT_wait_all': {'mean': 0.418, 'CI_lower': 0.411, 'CI_upper': 0.426},
 'avg_PT_wait_matched': {'mean': 0.188, 'CI_lower': 0.183, 'CI_upper': 0.192},
 'avg_earnings_per_hour': {'mean': 22.609,
  'CI_lower': 22.543,
  'CI_upper': 22.676},
 'avg_driver_profit': {'mean': 142.837,
  'CI_lower': 142.386,
  'CI_upper': 143.289},
 'avg_rides_per_driver': {'mean': 7.165, 'CI_lower': 7.14, 'CI_upper': 7.189},
 'avg_working_prop': {'mean': 0.866, 'CI_lower': 0.863, 'CI_upper': 0.87},
 'avg_idle_time_per_ride': {'mean': 0.373,
  'CI_lower': 0.371,
  'CI_upper': 0.376},
 'idle_proportion': {'mean': 0.419, 'CI_lower': 0.417, 'CI_upper': 0.421}}

In [785]:
# make histogram plotting function 

def plot_histogram(data_vector, title, x_label, figure_name, save=False):
    ''' 
    - 'title', 'x_label', 'figure_name': strings
    - Set save=True to save the histogram as a png
    - 'figure_name' should be of the form "figure_name.png"
    '''
    plt.figure()
    plt.hist(data_vector, bins=20)
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel("Frequency")
    if save:
        plt.savefig(figure_name)
    plt.show()